In [35]:
import anndata as ad
import scanpy as sc
from pathlib import Path

# ============================
# Resolve paths
# ============================

# project root (where 3.label_transfer... is located)
proj_root = Path(".").resolve()

# model directory (fixed name after your modifications)
glue_dir = proj_root / "model" / "glue_run"

rna_path  = glue_dir / "rna_with_Xglue.h5ad"
atac_path = glue_dir / "atac_with_Xglue.h5ad"

print("[PATH] RNA:", rna_path)
print("[PATH] ATAC:", atac_path)

# ============================
# Load data
# ============================

rna  = ad.read_h5ad(rna_path)
atac = ad.read_h5ad(atac_path)

print("[INFO] Loaded:")
print("  RNA :", rna.shape)
print("  ATAC:", atac.shape)

# ============================
# Step 1: Fix RNA subCluster
# ============================

if "subCluster" not in rna.obs.columns:
    raise KeyError("RNA.obs does not contain 'subCluster'")

rna.obs["subCluster"] = rna.obs["subCluster"].astype(str)

mask_unknown = rna.obs["subCluster"] == "Unknown"
if "H2_annotation" in rna.obs.columns:
    rna.obs.loc[mask_unknown, "subCluster"] = rna.obs.loc[mask_unknown, "H2_annotation"].astype(str)
else:
    print("[WARN] H2_annotation not found — Unknown subclusters kept as Unknown.")

print("[INFO] subCluster updated:")
print(rna.obs["subCluster"].value_counts().head(20))

# ============================
# Check X_glue availability
# ============================

assert "X_glue" in rna.obsm, "RNA missing X_glue"
assert "X_glue" in atac.obsm, "ATAC missing X_glue"

print("[INFO] X_glue shapes:")
print("  RNA :", rna.obsm["X_glue"].shape)
print("  ATAC:", atac.obsm["X_glue"].shape)



[PATH] RNA: /io/mza/github/scBrain-TraitMap/model/glue_run/rna_with_Xglue.h5ad
[PATH] ATAC: /io/mza/github/scBrain-TraitMap/model/glue_run/atac_with_Xglue.h5ad
[INFO] Loaded:
  RNA : (2687, 293)
  ATAC: (1776, 543960)
[INFO] subCluster updated:
subCluster
OPC              120
Astro-CP-1        52
IN-MGE            49
EN-IT-L5-2-c2     47
EN-IT-L5-1-c2     46
EN-IT-L6-c1       46
EN-ET-L6-c3       44
EN-ET-L5-c3       43
EN-ET-L5-c4       42
EN-IT-L2-2-c5     41
EN-IT-L6-c2       41
EN-IT-L3-2-c4     40
EN-IT-L5-2-c4     40
Astro-IZ-2        40
EN-IT-L5-1-c1     40
EN-IT-L6-c7       40
EN-ET-SP-2-c5     39
EN-IT-L2/3-c2     39
EN-ET-SP-2-c7     39
EN-IT-L4/5-c1     39
Name: count, dtype: int64
[INFO] X_glue shapes:
  RNA : (2687, 50)
  ATAC: (1776, 50)


In [36]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_distances

# ============================================
# A) Parameters (expanded candidate pool)
# ============================================

k = 100                 # Larger K for more stable votes
radius_pct = 95         # Radius threshold (percentile)
min_margin = 2          # min(top1 - top2 votes)
min_votes = 5           # Minimum votes for the modal label
target_cover = 0.9      # Fraction of ATAC cells to assign in stage 1
min_quota = 5           # Minimum quota per cluster

# ============================================
# B) Load embeddings and labels
# ============================================

Xr = np.asarray(rna.obsm["X_glue"], dtype=np.float32)
Xa = np.asarray(atac.obsm["X_glue"], dtype=np.float32)

rna_lbl = rna.obs["subCluster"].astype(str).values
subclusters = pd.Index(sorted(pd.unique(rna_lbl)))

def finite_mask(X):
    ok = np.isfinite(X).all(axis=1)
    if not ok.all():
        print(f"[WARN] Non-finite embeddings detected: removing {np.size(ok)-ok.sum()} rows for KNN fitting")
    return ok

kr = finite_mask(Xr)
ka = finite_mask(Xa)

Xr_fit = Xr[kr]
Xa_fit = Xa[ka]
rna_lbl_fit = rna_lbl[kr]

# ============================================
# C) KNN voting in the SCGLUE space
# ============================================

print(f"[INFO] Fitting NearestNeighbors (k={k}, cosine metric)")
nbrs = NearestNeighbors(n_neighbors=k, metric="cosine").fit(Xr_fit)
dists, idx = nbrs.kneighbors(Xa_fit, return_distance=True)

votes = rna_lbl_fit[idx]  # (n_atac_local, k)

df_votes = pd.DataFrame(votes)
mode_lbl = df_votes.mode(axis=1)[0].astype(str).values
mode_cnt = (df_votes.values == mode_lbl[:, None]).sum(axis=1)

# top1 and top2 vote counts
def top2_counts(row):
    vc = pd.value_counts(row)
    if len(vc) == 1:
        return np.array([vc.iloc[0], 0], dtype=int)
    return vc.iloc[:2].to_numpy()

top2_cnt = np.vstack([top2_counts(v) for v in votes])
margin = top2_cnt[:, 0] - top2_cnt[:, 1]

# radius threshold
rad = np.median(dists, axis=1)
tau = np.percentile(rad, radius_pct)

ok = (rad <= tau) & (margin >= min_margin) & (mode_cnt >= min_votes)
print(f"[INFO] Stage-1 screening: {ok.sum()} / {Xa_fit.shape[0]} passed (tau={tau:.4f})")

# ============================================
# D) Candidate pool + quotas
# ============================================

candidates = {sc: [] for sc in subclusters}

for a_local in np.where(ok)[0]:
    lab = mode_lbl[a_local]
    mask_same = (votes[a_local] == lab)
    if not np.any(mask_same):
        continue
    cost = dists[a_local][mask_same].mean()
    conf = mode_cnt[a_local] / float(k)
    candidates[lab].append((cost, conf, a_local))

for sc in subclusters:
    candidates[sc].sort(key=lambda x: x[0])

# total target
target_total = int(target_cover * Xa.shape[0])

# proportional quotas
rna_counts = pd.Series(rna_lbl).value_counts().reindex(subclusters, fill_value=0)
quota = (rna_counts / rna_counts.sum() * target_total).round().astype(int)

# enforce minimum + cap by availability
for sc in subclusters:
    avail = len(candidates[sc])
    if avail == 0:
        quota.loc[sc] = 0
    else:
        quota.loc[sc] = min(max(min_quota, quota.loc[sc]), avail)

print("[INFO] Quotas (after adjustment):")
print(quota.to_string())

# ============================================
# E) Greedy round-robin assignment
# ============================================

assigned = {sc: [] for sc in subclusters}
used = np.zeros(Xa_fit.shape[0], dtype=bool)

round_take = 100
rounds = 0

while True:
    rounds += 1
    progress = 0
    for sc in subclusters:
        need = quota.loc[sc] - len(assigned[sc])
        if need <= 0:
            continue
        picks = []
        for cost, conf, a_local in candidates[sc]:
            if used[a_local]:
                continue
            picks.append((a_local, cost, conf))
            if len(picks) >= min(round_take, need):
                break
        if picks:
            for a_local, cost, conf in picks:
                assigned[sc].append((a_local, cost, conf))
                used[a_local] = True
            progress += len(picks)

    print(f"[INFO] Round {rounds}: assigned {progress}")
    if progress == 0:
        print("[STOP] No more candidates to assign")
        break
    if sum(len(v) for v in assigned.values()) >= quota.sum():
        print("[STOP] Reached total quota")
        break

# ============================================
# F) Write results back (v2) + relaxed + fallback
# ============================================

local2global = np.where(ka)[0]
n_all = atac.n_obs

lab_v2  = np.array(["unassigned"] * n_all, dtype=object)
conf_v2 = np.full(n_all, np.nan, dtype=np.float32)
stage_v2 = np.array(["unassigned"] * n_all, dtype=object)

# Stage 1 assignments
for sc, arr in assigned.items():
    for a_local, cost, conf in arr:
        g = local2global[a_local]
        lab_v2[g] = sc
        conf_v2[g] = conf
        stage_v2[g] = "strict"

# Stage 2 relaxed mode
unassigned_global = np.where(lab_v2 == "unassigned")[0]
g2l = {g: l for l, g in enumerate(local2global)}

tau_relax = np.percentile(rad, 99)
ok_relax = (rad <= tau_relax) & (margin >= 1)

print(f"[INFO] Stage-2 relaxed reassignment: {len(unassigned_global)} cells")

for g in unassigned_global:
    l = g2l.get(g, None)
    if l is None:
        continue
    if ok_relax[l]:
        lab_v2[g]  = mode_lbl[l]
        conf_v2[g] = mode_cnt[l] / float(k)
        stage_v2[g] = "relaxed"

# Stage 3 fallback to RNA centroids
still_un = np.where(lab_v2 == "unassigned")[0]

if len(still_un) > 0:
    print(f"[INFO] Stage-3 fallback: {len(still_un)} cells")
    centroids = {sc: Xr[rna_lbl == sc].mean(axis=0) for sc in subclusters}
    C = np.vstack([centroids[sc] for sc in subclusters]).astype(np.float32)
    D = cosine_distances(Xa[still_un], C)
    best = D.argmin(axis=1)
    for i, g in enumerate(still_un):
        lab_v2[g] = subclusters[best[i]]
        conf_v2[g] = np.nan
        stage_v2[g] = "fallback"

# Save columns
atac.obs["assigned_subCluster_v2"] = pd.Categorical(
    lab_v2, categories=list(subclusters) + ["unassigned"]
)
atac.obs["assign_confidence_v2"] = conf_v2
atac.obs["assign_stage_v2"] = pd.Categorical(
    stage_v2,
    categories=["strict", "relaxed", "fallback", "unassigned"]
)

# Summary print
print("\n[SUMMARY v2: labels]")
print(atac.obs["assigned_subCluster_v2"].value_counts())

print("\n[SUMMARY v2: assignment stages]")
print(atac.obs["assign_stage_v2"].value_counts())


[INFO] Fitting NearestNeighbors (k=100, cosine metric)
[INFO] Stage-1 screening: 1386 / 1776 passed (tau=0.5368)
[INFO] Quotas (after adjustment):
Astro-1           0
Astro-CP-1       31
Astro-CP-2        0
Astro-IZ-1        0
Astro-IZ-2       24
EN-ET-L5-c1       4
EN-ET-L5-c2       9
EN-ET-L5-c3       1
EN-ET-L5-c4      14
EN-ET-L6-c1       1
EN-ET-L6-c2      15
EN-ET-L6-c3      22
EN-ET-L6-c4       0
EN-ET-L6-c5       0
EN-ET-L6-c6       2
EN-ET-SP-1-c1     2
EN-ET-SP-1-c2     2
EN-ET-SP-1-c3     0
EN-ET-SP-1-c4     0
EN-ET-SP-1-c5     0
EN-ET-SP-2-c1    21
EN-ET-SP-2-c2     2
EN-ET-SP-2-c3    23
EN-ET-SP-2-c4     1
EN-ET-SP-2-c5    14
EN-ET-SP-2-c6     0
EN-ET-SP-2-c7    23
EN-ET-SP-3-c1    21
EN-ET-SP-3-c2    23
EN-ET-SP-3-c3     5
EN-ET-SP-3-c4     9
EN-ET-SP-3-c5     0
EN-IT-L2-1-c1     3
EN-IT-L2-1-c2     0
EN-IT-L2-1-c3    20
EN-IT-L2-1-c4     1
EN-IT-L2-2-c1     3
EN-IT-L2-2-c2     5
EN-IT-L2-2-c3    12
EN-IT-L2-2-c4     0
EN-IT-L2-2-c5    14
EN-IT-L2/3-c1     7
EN-IT-L2/3-c2

In [37]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path
from collections import defaultdict
import time

# ============================================
# A) Parameters (editable)
# ============================================

OUTDIR               = Path("./ldsc_peaksets")  # Output directory
BLACKLIST_BED        = None      # Optional: path to blacklist BED
BASE_MIN_PREV        = 0.05      # Relaxed threshold: within-cluster prevalence
BASE_MIN_LOG2FC      = 0.5       # Relaxed log2FC cutoff
FALLBACK_MIN_PREV    = 0.02      # Fallback minimum prevalence
FALLBACK_MIN_LOG2FC  = 0.0       # Fallback minimum log2FC
TARGET_MIN_PER_TYPE  = 200       # Minimum peaks per cluster
TOP_K_PER_TYPE       = None      # Optional upper cap
EPS                  = 1e-9

t0 = time.time()
print(">>> Re-picking specific peaks with relaxed thresholds")

# ============================================
# B) Utility functions
# ============================================

def parse_peak_name(name: str):
    """Convert 'chr:start-end' into (chrom, start, end)."""
    s = name.replace(":", "-")
    chrom, start, end = s.split("-")
    return chrom, int(start), int(end)

def js_divergence(P, Q, eps=1e-12):
    """Jensen-Shannon divergence for peak specificity."""
    P = np.clip(P, eps, 1.0)
    P /= P.sum(axis=0, keepdims=True)

    Q = np.clip(Q, eps, 1.0)
    Q /= Q.sum(axis=0, keepdims=True)

    M = 0.5 * (P + Q)

    def _kl(A, B):
        return np.sum(A * (np.log(A) - np.log(B)), axis=0)

    return 0.5 * _kl(P, M) + 0.5 * _kl(Q, M)

# ============================================
# C) Prepare matrices
# ============================================

if "assigned_subCluster_v2" not in atac.obs:
    raise ValueError("ATAC AnnData must contain 'assigned_subCluster_v2'.")

X = atac.layers.get("counts", atac.X)
if not sp.issparse(X):
    X = sp.csr_matrix(X)

X = X.tocsr()

# Binarized accessibility
X_bin = X.copy()
X_bin.data = np.ones_like(X_bin.data)

# Optionally apply blacklist removal
valid_peak_mask = np.ones(atac.n_vars, dtype=bool)
kept_var = atac.var.copy()

# Background: peaks accessible in ≥1 cell
bg_mask = (X_bin.sum(axis=0) > 0).A1
X_bin = X_bin[:, bg_mask]
kept_var = kept_var.iloc[bg_mask]

n_cells = atac.n_obs
overall_prev = np.asarray(X_bin.sum(axis=0)).ravel() / float(n_cells)

# ============================================
# D) Compute per-cluster prevalence + JSD
# ============================================

labels = atac.obs["assigned_subCluster_v2"].astype(str).values
types, inv = np.unique(labels, return_inverse=True)

# Build cluster × cell assignment matrix
rows = inv
cols = np.arange(atac.n_obs, dtype=int)
G = sp.coo_matrix((np.ones_like(cols), (rows, cols)),
                  shape=(len(types), atac.n_obs)).tocsr()

group_sum = G @ X_bin
group_sizes = np.asarray(G.sum(axis=1)).ravel().astype(float)

prevalence = group_sum.multiply(1.0 / group_sizes[:, None]).astype(float).A

P = prevalence.copy()
P /= (P.sum(axis=0, keepdims=True) + EPS)

Q = np.tile(overall_prev / (overall_prev.sum() + EPS), (len(types), 1))

JSD = js_divergence(P, Q)

print(f"[INFO] n_clusters={len(types)}, n_peaks={X_bin.shape[1]}")

# ============================================
# E) Select peaks per cluster (relaxed + fallback)
# ============================================

type_specific = {}
total = 0

for i, t in enumerate(types):
    prev_t = prevalence[i]
    other_prev = (prevalence.sum(axis=0) - prev_t) / max(len(types) - 1, 1)
    log2fc = np.log2((prev_t + EPS) / (other_prev + EPS))

    # 1) Initial relaxed filter
    base_mask = (prev_t >= BASE_MIN_PREV) & (log2fc >= BASE_MIN_LOG2FC)
    idx = np.where(base_mask)[0]

    # 2) Fallback expansion (use top JSD peaks)
    if idx.size < TARGET_MIN_PER_TYPE:
        fb_mask = (prev_t >= FALLBACK_MIN_PREV) & (log2fc >= FALLBACK_MIN_LOG2FC)
        idx_fb = np.where(fb_mask)[0]

        if idx_fb.size > 0:
            order_fb = np.argsort(-JSD[idx_fb])
            need = TARGET_MIN_PER_TYPE - idx.size
            extra = idx_fb[order_fb[:max(0, need)]]
            idx = np.union1d(idx, extra)

    # 3) Optional cap
    if TOP_K_PER_TYPE is not None and idx.size > TOP_K_PER_TYPE:
        order = np.argsort(-JSD[idx])
        idx = idx[order[:TOP_K_PER_TYPE]]

    type_specific[t] = idx
    total += idx.size

print(f"[SUMMARY] Total specific peaks across clusters: {total}")

# ============================================
# F) Write BED files + summary
# ============================================

OUTDIR.mkdir(parents=True, exist_ok=True)

# Background union peaks
bg_bed = OUTDIR / "background_union_open_peaks.bed"
with open(bg_bed, "w") as f:
    for name in kept_var.index:
        chrom, start, end = parse_peak_name(name)
        f.write(f"{chrom}\t{start}\t{end}\n")

rows = []
for t, idx in type_specific.items():
    fname = OUTDIR / f"{t.replace('/', '_').replace(' ', '_')}.specific_peaks.bed"
    with open(fname, "w") as f:
        for name in kept_var.index[idx]:
            chrom, start, end = parse_peak_name(name)
            f.write(f"{chrom}\t{start}\t{end}\n")

    rows.append((t, len(idx), str(fname)))

df_summary = pd.DataFrame(
    rows,
    columns=["subCluster", "n_specific_peaks", "bed"]
).sort_values("n_specific_peaks", ascending=False)

tsv = OUTDIR / "specific_peak_counts.tsv"
df_summary.to_csv(tsv, sep="\t", index=False)

print("\n[PER-CLUSTER COUNTS]")
print(df_summary.to_string(index=False))
print(f"\nSummary saved to: {tsv}")
print(f"Done. Elapsed: {time.time() - t0:.1f}s")


>>> Re-picking specific peaks with relaxed thresholds
[INFO] n_clusters=74, n_peaks=524709
[SUMMARY] Total specific peaks across clusters: 2133841

[PER-CLUSTER COUNTS]
   subCluster  n_specific_peaks                                            bed
  EN-ET-L5-c2             89146   ldsc_peaksets/EN-ET-L5-c2.specific_peaks.bed
EN-IT-L5-1-c2             86656 ldsc_peaksets/EN-IT-L5-1-c2.specific_peaks.bed
EN-IT-L3-1-c3             84484 ldsc_peaksets/EN-IT-L3-1-c3.specific_peaks.bed
EN-IT-L5-2-c2             81579 ldsc_peaksets/EN-IT-L5-2-c2.specific_peaks.bed
EN-IT-L2-2-c5             72133 ldsc_peaksets/EN-IT-L2-2-c5.specific_peaks.bed
  EN-IT-L4-c2             72020   ldsc_peaksets/EN-IT-L4-c2.specific_peaks.bed
EN-IT-L5-1-c4             64609 ldsc_peaksets/EN-IT-L5-1-c4.specific_peaks.bed
EN-IT-L3-2-c3             64529 ldsc_peaksets/EN-IT-L3-2-c3.specific_peaks.bed
EN-IT-L5-2-c4             63340 ldsc_peaksets/EN-IT-L5-2-c4.specific_peaks.bed
  EN-ET-L5-c4             62614   ldsc_pe

In [38]:
import pandas as pd
from pathlib import Path

# --- Try v2 directory first ---
DIR_CANDIDATES = [
    Path("model/glue_run/ldsc_peaksets_v2"),
    Path("model/glue_run/ldsc_peaksets"),
    Path("ldsc_peaksets_v2"),
    Path("ldsc_peaksets")
]

OUTDIR = None
TSV = None

for d in DIR_CANDIDATES:
    tsv = d / "specific_peak_counts.tsv"
    if tsv.exists():
        OUTDIR = d
        TSV = tsv
        break

if OUTDIR is None:
    raise FileNotFoundError(
        "Cannot find specific_peak_counts.tsv in any known directory.\n"
        "Checked:\n" + "\n".join(str(d) for d in DIR_CANDIDATES)
    )

print(f"[INFO] Using peakset directory: {OUTDIR}")

# Load table
df = pd.read_csv(TSV, sep="\t")

# ---- Summary 1: sorted counts ----
print("Specific peak counts per subCluster (descending):")
print(
    df.sort_values("n_specific_peaks", ascending=False)
      .to_string(index=False)
)

# ---- Summary 2: whether each class meets the target ----
CUT = 1000  # threshold for "sufficient" number of peaks

df_status = df.assign(
    status=lambda d: d["n_specific_peaks"].ge(CUT).map({True: "OK", False: "LOW"})
)

print(f"\nCount of subClusters by status (OK >= {CUT}, LOW < {CUT}):")
print(df_status.groupby("status").size())


[INFO] Using peakset directory: ldsc_peaksets
Specific peak counts per subCluster (descending):
   subCluster  n_specific_peaks                                            bed
  EN-ET-L5-c2             89146   ldsc_peaksets/EN-ET-L5-c2.specific_peaks.bed
EN-IT-L5-1-c2             86656 ldsc_peaksets/EN-IT-L5-1-c2.specific_peaks.bed
EN-IT-L3-1-c3             84484 ldsc_peaksets/EN-IT-L3-1-c3.specific_peaks.bed
EN-IT-L5-2-c2             81579 ldsc_peaksets/EN-IT-L5-2-c2.specific_peaks.bed
EN-IT-L2-2-c5             72133 ldsc_peaksets/EN-IT-L2-2-c5.specific_peaks.bed
  EN-IT-L4-c2             72020   ldsc_peaksets/EN-IT-L4-c2.specific_peaks.bed
EN-IT-L5-1-c4             64609 ldsc_peaksets/EN-IT-L5-1-c4.specific_peaks.bed
EN-IT-L3-2-c3             64529 ldsc_peaksets/EN-IT-L3-2-c3.specific_peaks.bed
EN-IT-L5-2-c4             63340 ldsc_peaksets/EN-IT-L5-2-c4.specific_peaks.bed
  EN-ET-L5-c4             62614   ldsc_peaksets/EN-ET-L5-c4.specific_peaks.bed
EN-IT-L3-2-c1             58233 lds

In [39]:
import pandas as pd

print("First 20 peak names:")
print(pd.Series(atac.var_names[:20]).to_string())


First 20 peak names:
0       chr1:11655233-11655732
1     chr4:186616224-186616723
2     chr5:165520753-165521252
3     chr3:107593828-107594327
4      chr14:35904431-35904930
5     chr3:116058368-116058867
6     chr2:145824340-145824839
7      chr18:14947455-14947954
8     chr2:201987681-201988180
9     chr6:127855752-127856251
10      chr2:55226774-55227273
11     chr12:94501903-94502402
12     chr11:27772326-27772825
13        chr5:4216337-4216836
14       chr20:5585267-5585766
15      chr2:58359578-58360077
16      chr1:65956400-65956899
17         chr20:424793-425292
18      chr4:99236278-99236777
19      chr2:16803410-16803909


In [40]:
import pandas as pd

def add_peak_coords(atac, overwrite=False):
    """
    Parse ATAC var_names formatted as 'chrX:start-end', and add
    chrom / chromStart / chromEnd columns into atac.var.
    """

    idx = pd.Index(atac.var_names.astype(str))

    # Parse "chrX:start-end"
    coords = (
        idx.to_series(index=idx, name="peak")
        .str.extract(r'^(chr[^:]+):(\d+)-(\d+)$')
    )
    coords.columns = ["chrom", "chromStart", "chromEnd"]

    # Convert to numeric
    coords["chromStart"] = pd.to_numeric(coords["chromStart"], errors="coerce")
    coords["chromEnd"]   = pd.to_numeric(coords["chromEnd"],   errors="coerce")

    # Respect user overwrite option
    to_join = coords.copy()
    if not overwrite:
        for col in ["chrom", "chromStart", "chromEnd"]:
            if col in atac.var.columns:
                to_join = to_join.drop(columns=[col])

    # Merge
    atac.var = atac.var.join(to_join)

    # Report bad peaks
    bad = to_join["chrom"].isna().sum() if "chrom" in to_join else 0
    if bad > 0:
        print(f"[WARN] {bad} peak names could not be parsed (unexpected format).")
    else:
        print("[OK] All peak names parsed successfully!")


# ==== Example usage ====
add_peak_coords(atac,overwrite=True)


[OK] All peak names parsed successfully!


In [41]:
print(atac.var[["chrom", "chromStart", "chromEnd"]].head())

                          chrom  chromStart   chromEnd
chr1:11655233-11655732     chr1    11655233   11655732
chr4:186616224-186616723   chr4   186616224  186616723
chr5:165520753-165521252   chr5   165520753  165521252
chr3:107593828-107594327   chr3   107593828  107594327
chr14:35904431-35904930   chr14    35904431   35904930


In [42]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path
from glob import glob

# ======================================================
# A) Configure paths and parameters
# ======================================================

outdir = Path("./ldsc_peaksets")   # Output directory
tsv = outdir / "specific_peak_counts.tsv"

TARGET_MIN = 1000                 # Minimum peaks per cluster after fallback
FALLBACK_MIN_PREV = 0.01          # Fallback threshold: min within-cluster prevalence
FALLBACK_MIN_LOG2FC = -0.2        # Fallback threshold: min log2FC
TOP_K_PER_TYPE = 30000            # Per-cluster upper limit (None for unlimited)

# ======================================================
# B) Load or reconstruct peak count table
# ======================================================

def load_counts_table(outdir, tsv):
    """Load specific_peak_counts.tsv or reconstruct from BED files."""
    if tsv.exists():
        df = pd.read_csv(tsv, sep="\t")
    else:
        rows = []
        for bed in glob(str(outdir / "*.specific_peaks.bed")):
            name = Path(bed).name.replace(".specific_peaks.bed", "")
            n = sum(1 for _ in open(bed))
            rows.append({"subCluster": name, "n_specific_peaks": n, "bed": bed})
        df = pd.DataFrame(rows)

    # Normalize column names
    if "assigned_subCluster" in df.columns and "subCluster" not in df.columns:
        df = df.rename(columns={"assigned_subCluster": "subCluster"})

    if "subCluster" not in df.columns:
        raise ValueError("Missing column: subCluster")

    if "n_specific_peaks" not in df.columns:
        raise ValueError("Missing column: n_specific_peaks")

    return df


df_cnt = load_counts_table(outdir, tsv).copy()
df_cnt = df_cnt.set_index("subCluster").sort_index()

print("Current specific peak counts (first 10):")
print(df_cnt.head(10))

# ======================================================
# C) Compute prevalence, log2FC, and JSD
# ======================================================

X = atac.layers.get("counts", atac.X)
if not sp.isspmatrix_csr(X):
    X = sp.csr_matrix(X)

labels = atac.obs["assigned_subCluster_v2"].astype(str).values
groups = pd.Index(sorted(np.unique(labels)))
gmap = {g: i for i, g in enumerate(groups)}

# One-hot encoding for clusters
row = np.arange(atac.n_obs)
col = np.vectorize(gmap.get)(labels)
G = sp.csr_matrix((np.ones_like(row), (row, col)),
                  shape=(atac.n_obs, len(groups)))

# Binarized accessibility
X_bin = (X > 0)
bg_mask = (X_bin.sum(axis=0) > 0).A1
Xb = X_bin[:, bg_mask]

n_cells = atac.n_obs
group_sizes = np.asarray(G.sum(axis=0)).ravel()

prev = (G.T @ Xb).astype(np.float32)
prev = prev.multiply(1.0 / (group_sizes[:, None] + 1e-9)).toarray()

overall_prev = (Xb.sum(axis=0).A1 / n_cells).astype(np.float32)
mean_other = (prev.sum(axis=0, keepdims=True) - prev) / max(len(groups) - 1, 1)

eps = 1e-6
log2fc = np.log2((prev + eps) / (mean_other + eps))

# Jensen-Shannon Divergence: cluster distribution vs. uniform
P = prev / np.clip(prev.sum(axis=0, keepdims=True), 1e-12, None)
Q = np.full((len(groups), 1), 1.0 / len(groups), dtype=np.float32)
M = 0.5 * (P + Q)

def _kl(a, b):
    a = np.clip(a, 1e-12, 1.0)
    b = np.clip(b, 1e-12, 1.0)
    return (a * np.log2(a / b)).sum(axis=0)

JSD = 0.5 * _kl(P, M) + 0.5 * _kl(Q, M)

# Extract peak coordinates aligned with bg_mask
kept = atac.var.loc[bg_mask, ["chrom", "chromStart", "chromEnd"]].reset_index(drop=True)

# ======================================================
# D) Supplement insufficient clusters
# ======================================================

added_summary = []

for gi, g in enumerate(groups):
    old_n = int(df_cnt.loc[g, "n_specific_peaks"]) if g in df_cnt.index else 0
    if old_n >= TARGET_MIN:
        continue

    # Relaxed candidate filter
    cand = np.where((prev[gi] >= FALLBACK_MIN_PREV) &
                    (log2fc[gi] >= FALLBACK_MIN_LOG2FC))[0]

    if cand.size == 0:
        continue

    quota = min(TARGET_MIN, TOP_K_PER_TYPE)
    need = max(0, quota - old_n)
    if need <= 0:
        continue

    # Select highest JSD peaks
    order = np.argsort(-JSD[cand])[:need]
    take = cand[order]

    # Merge with existing peaks
    bed_path = outdir / f"{g.replace('/', '_')}.specific_peaks.bed"
    if bed_path.exists():
        old_df = pd.read_csv(bed_path, sep="\t", header=None,
                             names=["chrom", "start", "end"])
    else:
        old_df = pd.DataFrame(columns=["chrom", "start", "end"])

    add_df = kept.iloc[take].rename(
        columns={"chrom": "chrom", "chromStart": "start", "chromEnd": "end"}
    )

    new_df = pd.concat([old_df, add_df], axis=0, ignore_index=True).drop_duplicates()
    new_df.to_csv(bed_path, sep="\t", header=False, index=False)

    added_summary.append((g, old_n, new_df.shape[0]))

# ======================================================
# E) Print summary of additions
# ======================================================

if added_summary:
    print("\nSupplemented peaks (cluster, old_n → new_n):")
    for g, oldn, newn in added_summary:
        print(f"{g:25s}   {oldn:6d} → {newn:6d}")
else:
    print("\nAll clusters meet the minimum requirement; no fallback needed.")

# ======================================================
# F) Recompute count table and write TSV
# ======================================================

rows = []
for bed in sorted(outdir.glob("*.specific_peaks.bed")):
    name = bed.name.replace(".specific_peaks.bed", "")
    n = sum(1 for _ in open(bed))
    rows.append({"subCluster": name, "n_specific_peaks": n, "bed": str(bed)})

df_new = pd.DataFrame(rows).sort_values("n_specific_peaks", ascending=False)
df_new.to_csv(tsv, sep="\t", index=False)

print("\nUpdated specific peak counts (first 20):")
print(df_new.head(20).to_string(index=False))

print(f"\nSaved updated table: {tsv}")


Current specific peak counts (first 10):
             n_specific_peaks                                           bed
subCluster                                                                 
Astro-1                  5550      ldsc_peaksets/Astro-1.specific_peaks.bed
Astro-CP-1               3823   ldsc_peaksets/Astro-CP-1.specific_peaks.bed
Astro-CP-2              13359   ldsc_peaksets/Astro-CP-2.specific_peaks.bed
Astro-IZ-1                976   ldsc_peaksets/Astro-IZ-1.specific_peaks.bed
Astro-IZ-2               1318   ldsc_peaksets/Astro-IZ-2.specific_peaks.bed
EN-ET-L5-c1             22197  ldsc_peaksets/EN-ET-L5-c1.specific_peaks.bed
EN-ET-L5-c2             89146  ldsc_peaksets/EN-ET-L5-c2.specific_peaks.bed
EN-ET-L5-c3             22570  ldsc_peaksets/EN-ET-L5-c3.specific_peaks.bed
EN-ET-L5-c4             62614  ldsc_peaksets/EN-ET-L5-c4.specific_peaks.bed
EN-ET-L6-c1             15981  ldsc_peaksets/EN-ET-L6-c1.specific_peaks.bed

Supplemented peaks (cluster, old_n → new_n):
A